# Intermediate coding tutorial on foundations of interoperating databases 

by Jordan Fuhrman, Research Scientist at the University of Chicago. Generative AI used as a tool in the development of this notebook.

This notebook is intended to demonstrate fundamental concepts in utilizing interoperable databases.  In particular, we aim to motivate key ideas, including record linkage, matching algorithms, and blocking, among others.  

To demonstrate this, we'll use a synthetic database that has also been used for other MIDRC tutorials.  It can be downloaded at this link: [Data Download](https://data.midrc.org/discovery/P67C-YW55/).  Note, you must be signed in to the MIDRC Data Commons at [data.midrc.org](data.midrc.org) to access the file, the download button can be found at the bottom of the opened tab. When you download the file, please save it in your current working directory for this notebook.

Now, we will begin the tutorial by loading packages and loading the synthetic dataset.  

## Setup and Loading Data



In [ ]:
import hashlib
import math
import random
import re
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

Make sure that the path at the top of the next code block is correct for the location where the synthetic databse was saved. 

In [ ]:
DATA_PATH = Path("MIDRC-Learn_JNotebooks/MIDRC_Stratified_Sampling_Example_5000_Patient_Subset.xlsx")

EXPECTED_COLUMNS = [
    "submitter_id",
    "age_at_index",
    "covid19_positive",
    "ethnicity",
    "race",
    "sex",
    "modality",
]

def create_synthetic_mock_dataset(n=250, random_state=42):
    rng = np.random.default_rng(random_state)

    ethnicities = ["Not Hispanic or Latino", "Hispanic or Latino"]
    races = [
        "White",
        "Black or African American",
        "Asian",
        "American Indian or Alaska Native",
        "Native Hawaiian or Other Pacific Islander",
        "Other",
        "Unknown",
    ]
    sexes = ["Male", "Female"]
    modalities = [
        "CT",
        "MRI",
        "X-Ray",
        "Ultrasound",
        "PET",
        "Pathology",
        "Clinical Exam",
        "Lab Test",
    ]

    rows = []
    for i in range(1, n + 1):
        k = rng.integers(1, 4)
        modality_values = sorted(rng.choice(modalities, size=k, replace=False))

        rows.append({
            "submitter_id": f"case_{i:04d}",
            "age_at_index": int(np.clip(rng.normal(55, 18), 0, 90)),
            "covid19_positive": rng.choice(["Yes", "No"], p=[0.28, 0.72]),
            "ethnicity": rng.choice(ethnicities, p=[0.82, 0.18]),
            "race": rng.choice(races, p=[0.58, 0.18, 0.09, 0.03, 0.02, 0.06, 0.04]),
            "sex": rng.choice(sexes),
            "modality": ", ".join(modality_values),
        })

    return pd.DataFrame(rows)

if DATA_PATH.exists():
    df_raw = pd.read_excel(DATA_PATH)
else:
    print(f"{DATA_PATH} not found. Creating a synthetic dataset so the notebook can run.")
    df_raw = create_synthetic_mock_dataset()

missing = [c for c in EXPECTED_COLUMNS if c not in df_raw.columns]
if missing:
    raise ValueError(f"Dataset is missing required columns: {missing}")

df_raw = df_raw[EXPECTED_COLUMNS].copy()
df_raw.head()

## Basic standardization and normalization

An important point in linking data between interoperating systems is that some algorithms are sensitive to potential changes between the systems (e.g., variable formatting).  While this is not an issue in our data at this point since we'll be generating synthetic interoperating databases, we do include some functions here to demonstrate how normalization may be performed.

In [ ]:
def normalize_yes_no(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    if value in {"yes", "y", "true", "1", "positive"}:
        return "Yes"
    if value in {"no", "n", "false", "0", "negative"}:
        return "No"
    return str(value).strip()

def normalize_text(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    return value

def parse_modality_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        parts = value
    else:
        parts = str(value).split(",")
    return sorted({normalize_text(p) for p in parts if normalize_text(p)})

def normalize_dataset(df):
    out = df.copy()

    out["submitter_id"] = out["submitter_id"].astype(str).str.strip()
    out["age_at_index"] = pd.to_numeric(out["age_at_index"], errors="coerce").astype("Int64")
    out["covid19_positive"] = out["covid19_positive"].apply(normalize_yes_no)
    out["ethnicity"] = out["ethnicity"].apply(normalize_text)
    out["race"] = out["race"].apply(normalize_text)
    out["sex"] = out["sex"].apply(normalize_text)
    out["modality_list"] = out["modality"].apply(parse_modality_list)
    out["modality_norm"] = out["modality_list"].apply(lambda xs: ", ".join(xs))

    return out

df = normalize_dataset(df_raw)
df.head()

In [ ]:
print("Rows:", len(df))
print("\nMissing values:")
display(df[EXPECTED_COLUMNS].isna().sum())

print("\nUnique values:")
for col in ["covid19_positive", "ethnicity", "race", "sex"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

## Simulating interoperating systems

In this step, we generate 2 interoperating systems (System A and System B) from our synthetic database.  Specifically, we add a column (source_system) to the dataframe that assigns each case to either or both of the interoperating systems.  Note, you can adjust the overlap rate to see more overlap between the synthetic systems, though we recommend keeping it relatively small; large overlap will make later stages take much longer time to complete. 

Additionally, note that each case row has an element "true_case_id".  This will serve as our truth/reference label to evaluate performance of the linkage methods demonstrated later in this notebook.  Also note that this does not address potential issues of a patient having multiple IDs within the same system; we leave that outside the scope of this demonstration but it should be considered for practical implementation.

In [ ]:
def split_into_mock_systems(
    df,
    systems=("System A", "System B"),
    overlap_rate=0.10,
    random_state=42
):
    '''
    Create simulated source-system records from a single original table.

    Each original case appears in at least one system.
    Some cases appear in multiple systems, creating true cross-system matches.
    '''
    rng = np.random.default_rng(random_state)
    rows = []

    for _, row in df.iterrows():
        included = {rng.choice(systems)}

        for system in systems:
            if system not in included and rng.random() < overlap_rate:
                included.add(system)

        for system in sorted(included):
            new_row = row.copy()
            new_row["source_system"] = system
            new_row["true_case_id"] = row["submitter_id"]
            new_row["local_record_id"] = f"{system.replace(' ', '_').lower()}_{row['submitter_id']}"
            rows.append(new_row)

    return pd.DataFrame(rows).reset_index(drop=True)

systems_df = split_into_mock_systems(df, systems=("System A", "System B"), overlap_rate=0.10)

print("Original cases:", df["submitter_id"].nunique())
print("Simulated source-system rows:", len(systems_df))
display(systems_df["source_system"].value_counts())
systems_df.head()

## Introduce system-specific differences

An important point of interoperating systems is that, in many cases, there will be some variation for a matching case between the two systems. For example, consider a patient who has moved across the country and had appointments at interoperating hospital systems.  We want to be able to successfully identify that this is the same patient in both systems, but they will have likely changed addresses or grown older, among other potential differences.  We need to simulate such scenarios, thus we incorporate missingness and small data perturbations between the 2 systems.

In [ ]:
def introduce_system_variation(df, random_state=42):
    '''
    Introduce realistic system-specific variation:
    - Small age shifts
    - Missing categorical values
    - Slightly different COVID encodings
    - Dropped modalities
    - Extra modalities
    '''
    rng = np.random.default_rng(random_state)
    out = df.copy()

    possible_modalities = sorted({
        modality
        for values in out["modality_list"]
        for modality in values
    })

    for idx in out.index:
        system = out.loc[idx, "source_system"]

        # Age may differ slightly across systems, especially if measured at different index dates.
        if rng.random() < 0.15:
            shift = rng.choice([-2, -1, 1, 2], p=[0.15, 0.35, 0.35, 0.15])
            if pd.notna(out.loc[idx, "age_at_index"]):
                out.loc[idx, "age_at_index"] = max(0, int(out.loc[idx, "age_at_index"]) + int(shift))

        # Some fields may be absent in one system.
        for col, prob in {
            "ethnicity": 0.05,
            "race": 0.05,
            "sex": 0.02,
            "covid19_positive": 0.04,
        }.items():
            if rng.random() < prob:
                out.loc[idx, col] = np.nan

        # System B occasionally uses lowercase yes/no labels.
        if system == "System B" and pd.notna(out.loc[idx, "covid19_positive"]) and rng.random() < 0.20:
            out.loc[idx, "covid19_positive"] = str(out.loc[idx, "covid19_positive"]).lower()

        # Modality lists may be incomplete or contain one extra recorded procedure.
        modalities = list(out.loc[idx, "modality_list"])

        if modalities and rng.random() < 0.20:
            modalities.remove(rng.choice(modalities))

        if possible_modalities and rng.random() < 0.08:
            candidate = rng.choice(possible_modalities)
            if candidate not in modalities:
                modalities.append(candidate)

        modalities = sorted(set(modalities))
        out.at[idx, "modality_list"] = modalities
        out.loc[idx, "modality_norm"] = ", ".join(modalities)
        out.loc[idx, "modality"] = ", ".join(modalities)

    out["covid19_positive"] = out["covid19_positive"].apply(normalize_yes_no)
    return out.reset_index(drop=True)

systems_df = introduce_system_variation(systems_df)

systems_df.sample(10, random_state=RANDOM_SEED)

## Create candidate pairs across systems

Now that we've finished creating the two simulated interoperating data sets, we can begin to explore the record linkage process.  The first thing that we will do is form create a dataframe of all possible pairs between the two interoperating systems as this is the most straightforward method to evaluating linkage between the datasets.  Note, however, that creating pairs between all possible candidates is computationally expensive and time-consuming; depending on the computing resources used to run this notebook and earlier parameter settings (e.g., overlap_rate of the systems), the two databases of ~2700 cases each can take on the order of 10-100 minutes to complete this code block (even after the progress bar is completed, there are multiple time-consuming steps).  Consider that modern databases are typically much larger, and thus comparing every potential pair for linkage may not be practical in real-world scenarios.

Ultimately, this section systematically constructs a dataframe comparing all cases in System A with all cases in System B.  Each row in the resulting dataframe represents a potential matched pair, i.e., the case from System A and the case from System B are the same, so in the next section, we'll evaluate different ways of distinguishing whether or not a pair candidate indeed represents a matched pair or if the pair comes from two different cases.  We also assign store the truth/reference case ID for both systems (in the true_case_id_l and true_case_id_r variables), which we will utilize for performance evaluation.

In [ ]:
from tqdm.notebook import tqdm

def make_cross_system_pairs(df, system_col="source_system"):

    left_rows = []
    right_rows = []

    total_comparisons = math.comb(len(df), 2)

    for left_idx, right_idx in tqdm(combinations(df.index, 2), total=total_comparisons, desc="Generating candidate pairs"):
        if df.loc[left_idx, system_col] == df.loc[right_idx, system_col]:
            continue
        left_rows.append(df.loc[left_idx])
        right_rows.append(df.loc[right_idx])

    left = pd.DataFrame(left_rows).add_suffix("_l").reset_index(drop=True)
    right = pd.DataFrame(right_rows).add_suffix("_r").reset_index(drop=True)

    return pd.concat([left, right], axis=1)

pairs = make_cross_system_pairs(systems_df)

print("Candidate cross-system pairs:", len(pairs))
pairs.head()

## Similarity functions for categorical, numeric, and list-valued fields

Now that we've generated the dataframe that allows for pairwise comparison of all of the cases in System A with all of the cases in System B, we can take a look at how to actually evaluate if the paired cases are a match.  

The most straightforward approach to this is to check if the field in System A has an exact match with the corresponding case in System B, which we utilize here for the race, ethnicity, sex, and covid19_positive variables.  However, an exact match may not always be appropriate; for example, it is possible for a patient's age to change between the time their case is created in System A and System B, so it would likely be detrimental to linkage performance to only consider pairs where there is an exact age match to be from the same case.  In this example, we use a similarity metric that decreases from 1 to 0 as the age variables for a pair are farther apart from each other, with a maximum difference (noted as 5 years in the code block below) indicating they are not a match (according to age).  Similarly, a case could have acquired additional imaging at one system that was not logged in the other, so our "modality" variable should not be evaluated for an exact match either.  Thus, we use the Jaccard similarity to evaluate if the overlap of modalities acquired for each case in a pair.

In evaluating the similarity for each variable of all pairs in our dataframe, we can compile the similarity measures into a single score.  Thus, we assign a weight to each variable similarity and add them together to provide a score between 0 and 1 for each pair.  

Finally, we also construct a column, "is_true_match", from our true_case_id_l and true_case_id_r variables that indicates if the pair is indeed a match and present the highest scoring matches according to the summed similarity values.  

In [ ]:
def exact_match(a, b):
    if pd.isna(a) or pd.isna(b):
        return 0.0
    return float(str(a).strip().lower() == str(b).strip().lower())

def age_similarity(a, b, max_difference=5):
    if pd.isna(a) or pd.isna(b):
        return 0.0
    diff = abs(int(a) - int(b))
    return max(0.0, 1.0 - diff / max_difference)

def jaccard_similarity(xs, ys):
    xs = set(xs or [])
    ys = set(ys or [])
    if not xs and not ys:
        return 1.0
    if not xs or not ys:
        return 0.0
    return len(xs & ys) / len(xs | ys)

def score_pairs(pairs):
    scored = pairs.copy()

    scored["age_sim"] = [
        age_similarity(a, b)
        for a, b in zip(scored["age_at_index_l"], scored["age_at_index_r"])
    ]

    for col in ["covid19_positive", "ethnicity", "race", "sex"]:
        scored[f"{col}_sim"] = [
            exact_match(a, b)
            for a, b in zip(scored[f"{col}_l"], scored[f"{col}_r"])
        ]

    scored["modality_sim"] = [
        jaccard_similarity(a, b)
        for a, b in zip(scored["modality_list_l"], scored["modality_list_r"])
    ]

    scored["weighted_score"] = (
        0.25 * scored["age_sim"]
        + 0.15 * scored["covid19_positive_sim"]
        + 0.15 * scored["ethnicity_sim"]
        + 0.15 * scored["race_sim"]
        + 0.10 * scored["sex_sim"]
        + 0.20 * scored["modality_sim"]
    )

    scored["is_true_match"] = scored["true_case_id_l"] == scored["true_case_id_r"]

    return scored

scored_pairs = score_pairs(pairs)

scored_pairs[
    [
        "local_record_id_l", "local_record_id_r",
        "source_system_l", "source_system_r",
        "age_sim", "race_sim", "sex_sim", "modality_sim",
        "weighted_score", "is_true_match"
    ]
].head(10)

## Threshold-based matching and evaluation

In the previous section, we obtained a similarity score for each pair of cases across System A and System B, thus in this section, we will take the next step to go from from the continuous similarity value to whether or not a pair of cases is actually called a match.  We can do this by applying a threshold at a certain similarity value, then calling everything above that threshold "matched" and everything below it "not matched".  

In this demonstration, we provide the confusion matrix values (true positives, false positive, false negatives, and true negatives) and subsequent metrics (precision, recall, and F1 score) to demonstrate how varying the similarity threshold can impact performance.  Note, we do NOT utilize accuracy here because the evaluation of match accuracy is heavily influence by the overlap of the two interoperating systems, whereas the other metrics are more robust to class imbalance.  For example, in a scenario where the overlap rate was only 0.01 (meaning 1% of cases were in both interoperating data bases), threshold that said there wasn't a single pair that matched would achieve an accuracy of 99%.  Whereas that same threshold in an interoperating system with 99% overlap rate would achieve a very low accuracy of 1%.  Because of this dependence, accuracy can be misleading for performance evaluation.

In practice, it can be difficult to determine which threshold value is the most optimal in an interoperating database because you likely won't have access to true case IDs as we do in this demonstration.  However, some approaches include creating a subset of gold standard data to determine the best threshold, utilizing a trusted source (e.g., an honest broker) that can receive confidential information from both interoperating systems and confirm match performance, or by experimentally determination via a simulation.  

In [ ]:
def evaluate_threshold(scored_pairs, threshold, score_col="weighted_score"):
    predicted = scored_pairs[score_col] >= threshold
    actual = scored_pairs["is_true_match"]

    tp = int((predicted & actual).sum())
    fp = int((predicted & ~actual).sum())
    fn = int((~predicted & actual).sum())
    tn = int((~predicted & ~actual).sum())

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision and recall and not np.isnan(precision) and not np.isnan(recall)
        else np.nan
    )

    return {
        "threshold": threshold,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

thresholds = np.concatenate([
    np.arange(0.30, 0.90, 0.05),
    np.arange(0.90, 1.001, 0.01)
])

evaluation = pd.DataFrame(
    evaluate_threshold(scored_pairs, threshold)
    for threshold in thresholds
)

evaluation

In reviewing the results, we see that recall remains high at low thresholds, but precision is poor because a large majority of pairs that were called matched were incorrect; at such a low threshold, even single matched demographic variables could result in a pair being called a match and results in a very high false positive rate.  As the threshold increases, we see the tradeoff between precision and recall, decreasing false positives at the expense of increased false negatives.  Typically, the F1 score will increase to a certain threshold value, then decrease.  In our sample data, we've only slightly perturbed the data between the interoperating systems and used a relatively discrete weighted sum of similarity scores, so you may see a range of equal F1 scores across high threshold values if you haven't adjusted earlier parameters.

## Blocking strategies

We've now completed the initial simulation, but there are several key areas where we can improve.  In particular, one of the most time-consuming and computationally expensive was developing the set of paired cases across interoperating systems; we evaluated every possible pairing of a case from each system, but there are many pairs where a thorough matching evaluation is unnecessary.  For example, consider a pair where one case is 32 years old and the other is 79 years old.  It seems extremely unlikely that these would belong to the same case, so we can use a comparison of simple characteristics to identify and rule out such potential pairs earlier on in the process.  This process is called blocking. 

If you have not changed any parameters from earlier cells, then Systems A and B each have approximately 2,500 cases, resulting in a paired dataset of 6,250,000 comparisons that have to be performed.  However, we can computationally reduce this through blocking.  In this example, we first create blocked keys, such as 5- and 10-year age ranges ("age_band_5" and "age_band10"), and combinations of other variables (e.g., a race_sex_block element might look like "Asian|Male" or "White|Female").  The most restrictive example in this demonstration is the "age_race_sex_block", which combines those three variables into a single combination.  Ultimately, we can utilize these blocked variable combinations as filters, only considering potential pairs if the have the same blocking key value.  

There are some important tradeoffs to blocking, though.  For example, consider a case where the patient's record in System A lists their age as 39 years old and the System B record lists them as 40 years old.  This pair of records would not be considered as a match because one belongs to the 30-39 year age band and the other in the 40-49 age band.  Thus, while we've constructed a more practical approach to identify potential matched pairs, it doesn't have quite the same flexibility as simply comparing the similarity of the variable between the two records.  

Upon running this code block, you will see the "Generating Candidate pairs:" message appear several times, as we generate pair candidates for each of the blocking keys in the dataset.  After this runs, we can compare the effectiveness of the different blocking strategies in reducing the number of pair candidates as well as the change in the recall of true matches across our interoperating systems.

In [ ]:
def age_band(age, width=10):
    if pd.isna(age):
        return "missing"
    age = int(age)
    lower = (age // width) * width
    upper = lower + width - 1
    return f"{lower:02d}-{upper:02d}"

blocked_df = systems_df.copy()
blocked_df["age_band_10"] = blocked_df["age_at_index"].apply(lambda x: age_band(x, width=10))
blocked_df["age_band_5"] = blocked_df["age_at_index"].apply(lambda x: age_band(x, width=5))
blocked_df["covid_sex_block"] = blocked_df["covid19_positive"].fillna("missing") + "|" + blocked_df["sex"].fillna("missing")
blocked_df["race_sex_block"] = blocked_df["race"].fillna("missing") + "|" + blocked_df["sex"].fillna("missing")
blocked_df["age_race_sex_block"] = (
    blocked_df["age_band_10"] + "|" + blocked_df["race"].fillna("missing") + "|" + blocked_df["sex"].fillna("missing")
)

def make_blocked_pairs(df, block_col):
    pieces = []
    for _, group in df.groupby(block_col, dropna=False):
        if group["source_system"].nunique() < 2:
            continue
        group_pairs = make_cross_system_pairs(group)
        if len(group_pairs):
            pieces.append(group_pairs)

    if not pieces:
        return pd.DataFrame()

    return pd.concat(pieces, ignore_index=True)

blocking_summary = []
all_true_matches = scored_pairs["is_true_match"].sum()

for block_col in ["age_band_10", "age_band_5", "covid_sex_block", "race_sex_block", "age_race_sex_block"]:
    blocked_pairs = make_blocked_pairs(blocked_df, block_col)
    blocked_scored = score_pairs(blocked_pairs) if len(blocked_pairs) else blocked_pairs

    retained_true_matches = int(blocked_scored["is_true_match"].sum()) if len(blocked_scored) else 0

    blocking_summary.append({
        "blocking_key": block_col,
        "candidate_pairs": len(blocked_pairs),
        "pair_reduction_vs_full": 1 - (len(blocked_pairs) / len(scored_pairs)),
        "true_matches_retained": retained_true_matches,
        "true_match_recall_before_scoring": retained_true_matches / all_true_matches if all_true_matches else np.nan,
    })

blocking_summary = pd.DataFrame(blocking_summary)
blocking_summary

## Hashing identifiers and linkage tokens

In addition to more practical computational improvements, we also need practical methods to evaluate matching performance in a real-world scenario.  Specifically, we know which candidates are true matches in our simulation, but a more realistic simulation would require that we do not know correct candidate matches a priori because a real set of interoperating databases may not be able to transmit unique ID information between systems.  The first method we'll review to address this issue is hashing.  Hashing refers to a deterministic transformation of an input value to an output representation with a fixed length.  In this section, we review 3 approaches to hashing and demonstrate how hashing can allow for performance evaluation, but also where hashing fails in true privacy preservation.

The first hashing approach is SHA-256, which converts our unique ID to an encoded ID.  More specifically, because hashing is a deterministic linkage between our input IDs and the output encoded ID, applying the same hash function to the IDs in both System A and System B will provide an encoded representation that does not directly reveal the unique ID; we could then transmit this encoded representation between systems to evaluate the matching performance.  However, simple hashing like this can be prone to cyberattacks/linkage between unique and encoded IDs if the hashing function and, for example, the format of unique IDs in each system are known.  To be clear, direct hashing as demonstrated here will obscure the unique ID, but it does not add any unknown information to the hash transformation.

The next step, then, is to add unknown information to transformation that is known to each system, but not necessarily known to outside sources.  This is called a salted hash.  We first demonstrate what linkage looks like if System A and System B each use their own salting approach.  For clarity, plain hashing may look something like "Case_001 --> encoded ID" whereas salted hashing would be "Case_001 + salt_A --> encoded ID".  In practice, the different interoperating systems could use different salting approaches, thus "Case_001 + salt_A" and "Case_001 + salt_B" would result in different output encoded IDs.  This is useful for privacy preservation, but makes linking cases in interoperating databases less straightforward.  

Alternatively, we can use a shared salt between the systems, thus requiring a nefarious party to also know the shared salting approach to uncover original IDs from the encoded IDs while also maintaining a relatively simple method of ID comparison for match evaluation.  This is performed in the 3rd example below.

In [ ]:
def sha256_hash(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()

def salted_hash(value, salt):
    return hashlib.sha256(f"{salt}|{value}".encode("utf-8")).hexdigest()

SALT_A = "training-salt-system-a"
SALT_B = "training-salt-system-b"
SHARED_SALT = "shared-linkage-salt-for-demo-only"

hashed_df = systems_df.copy()

hashed_df["submitter_id_hash_unsalted"] = hashed_df["submitter_id"].apply(sha256_hash)
hashed_df["submitter_id_hash_system_specific"] = [
    salted_hash(value, SALT_A if system == "System A" else SALT_B)
    for value, system in zip(hashed_df["submitter_id"], hashed_df["source_system"])
]
hashed_df["submitter_id_hash_shared"] = hashed_df["submitter_id"].apply(lambda value: salted_hash(value, SHARED_SALT))

hashed_df[
    [
        "source_system",
        "submitter_id",
        "submitter_id_hash_unsalted",
        "submitter_id_hash_system_specific",
        "submitter_id_hash_shared",
    ]
].head()

## Bloom filters for privacy-preserving approximate comparison

Importantly, hashing does not completely provide anonymization, only obfuscation.  Because hashing functions are deterministic, there remains a link between the original input ID and the encoded ID.  Further, we've already discussed above how mismatches between systems (e.g., misspelled names, address changes) can cause linkage quality to degrade, and hashes do not account for such errors (i.e., the encoded representation of a name spelled 2 different ways would not be linked).  Thus methods that account for this possibility via approximate matching rather than exact matching are desirable.

Bloom filters offer this possibility.  In brief, we first set a length of bits that will serve as our encoded representation, e.g., "0 0 0 0 0 0 0 0 0 0" is 10 bits.  Bloom filters first break down input data into equal-length representations; in this example, we break them down to representations of length=2, called bigrams, so the name "Smith" would be broken down to "^S"  "SM"  "MI"  "IT"  "TH"  "H$" with ^ and $ representing the start and end of a word, respectively.  The bigrams are then put through one or more hash functions that ultimately inform which bits in encoded representation will be changed to 1 and which will remain 0.  For example, suppose the hash function maps "Smith --> positions 1, 4, 9", the bloom filter would be "1 0 0 1 0 0 0 0 1 0", with positions 1, 4, and 9 being changed to a 1.  Importantly, the hash functions must be designed such that similar inputs will activate similar output bits, e.g., "Smyth --> positions 1, 5, 9" whereas a very different name would activate very different output bits e.g., "Jones --> positions 3, 8".  A comparison of these would look like: 
Smith --> 1 0 0 1 0 0 0 0 1 0
Smyth --> 1 0 0 0 1 0 0 0 1 0
Jones --> 0 0 1 0 0 0 0 1 0 0

Previously we utilized the Jaccard similarity as a continuous metric for evaluating the similarity between modalities used; we can re-use the Jaccard similarity here to compare the bit representations.  Thus, we can utilize Bloom filters in conjunction with hashing to get better approximate matching of IDs and allow for comparison of encoded ID representations that are obscured for potential transmission between systems while maintaining privacy.  

In [ ]:
def make_bigrams(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    if len(text) < 2:
        return [text]
    return [text[i:i+2] for i in range(len(text) - 1)]

def bloom_filter_encode(tokens, length=256, num_hashes=4, salt="demo-bloom-salt"):
    bits = np.zeros(length, dtype=int)

    for token in tokens:
        for i in range(num_hashes):
            digest = hashlib.sha256(f"{salt}|{i}|{token}".encode("utf-8")).hexdigest()
            position = int(digest, 16) % length
            bits[position] = 1

    return bits

def jaccard_similarity_bits(a, b):
    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)

    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    return intersection / union if union else 1.0

def quasi_identifier_string(row):
    return "|".join([
        str(age_band(row["age_at_index"], width=5)),
        str(row["covid19_positive"]),
        str(row["ethnicity"]),
        str(row["race"]),
        str(row["sex"]),
        str(row["modality_norm"]),
    ])

bloom_df = systems_df.copy()
bloom_df["qi_string"] = bloom_df.apply(quasi_identifier_string, axis=1)
bloom_df["qi_tokens"] = bloom_df["qi_string"].apply(make_bigrams)
bloom_df["qi_bloom_filter"] = bloom_df["qi_tokens"].apply(bloom_filter_encode)

bloom_pairs = make_cross_system_pairs(bloom_df)

bloom_pairs["bloom_jaccard_similarity"] = [
    jaccard_similarity_bits(a, b)
    for a, b in zip(bloom_pairs["qi_bloom_filter_l"], bloom_pairs["qi_bloom_filter_r"])
]

bloom_pairs["is_true_match"] = bloom_pairs["true_case_id_l"] == bloom_pairs["true_case_id_r"]

bloom_pairs[
    [
        "local_record_id_l",
        "local_record_id_r",
        "bloom_jaccard_similarity",
        "is_true_match",
    ]
].sort_values("bloom_jaccard_similarity", ascending=False).head(10)

## Differential privacy for aggregate sharing

While hashing and bloom filters offer an approach to provide privacy and comparison for individual IDs or even data elements.  However, there are other techniques that can allow nefarious parties to gain private information even if it is anonymized through bloom filters.  As an example, consider sources of aggregate data, such as the number of patients that live on a specific street, the number of pixels in a medical image that fall between a certain range of values, the rate of disease occurence within a system database.  Differential privacy offers a method to obscure such aggregate data while limiting what a nefarious party could infer/learn about any single record in the database.  

For example, consider in our System A dataset the number of cases which are disease-positive.  It is possible that through repeated and careful queries of the database that a nefarious party could learn whether or not a specific case is disease-positive.  So consider that, for example, the true number of disease-positive cases in System A is 351.  If there are repeated queries of this number for System A, instead of always reporting the exact number, we can add something called differential privacy, or a controlled random noise value.  So instead of always reporting 351 disease-positive cases, System A may report 350 the first time, then 354 the second, the 353 the third, etc.  In this way, we can obscure the exact number for the aggregate statistic while reporting useful information.  

It may be clear, however, that a large number of repeated requests for this aggregate statistic would allow a party to infer the mean value and, thus, identify the true value of the statistic.  To account for this, we introduce a privacy budget, epsilon, which is an adjustable parameter which indicates how much noise on average should be included in reporting the aggregate statistic.  In this demonstration, we compare a few epsilon values to show the impact of stronger or weaker privacy. In practice, one way to track the amount of information that has been released by repeated queries is to sum the epsilon values for each query; if a maximum privacy amount is reached, the system may stop responding to additional queries.  

To further emphasize how differential privacy may be important to interoperating systems, consider a user who wants to query System A and System B to identify the number of cases that fit a specified set of criteria.  To protect their data, each system may reported a differentially private aggregate statistic, allowing for cohort building, summary reporting, etc., while maintaining data privacy.  

In [ ]:
def laplace_mechanism(true_count, sensitivity=1.0, epsilon=1.0, random_state=None):
    rng = np.random.default_rng(random_state)
    scale = sensitivity / epsilon
    return true_count + rng.laplace(loc=0, scale=scale)

aggregate_counts = (
    systems_df
    .drop_duplicates("true_case_id")
    .groupby(["covid19_positive", "sex"], dropna=False)
    .size()
    .reset_index(name="true_count")
)

epsilon_values = [0.25, 0.5, 1.0, 2.0]

dp_rows = []
for _, row in aggregate_counts.iterrows():
    for epsilon in epsilon_values:
        noisy_count = laplace_mechanism(
            row["true_count"],
            sensitivity=1.0,
            epsilon=epsilon,
            random_state=RANDOM_SEED + int(row["true_count"] * 10) + int(epsilon * 100)
        )
        dp_rows.append({
            "covid19_positive": row["covid19_positive"],
            "sex": row["sex"],
            "epsilon": epsilon,
            "true_count": row["true_count"],
            "dp_noisy_count": round(max(0, noisy_count), 2),
        })

dp_counts = pd.DataFrame(dp_rows)
dp_counts